<a href="https://colab.research.google.com/github/oselumeseagbonrofo/small-llm-experiments/blob/main/Finetuning_SLMs_for_specific_domain_(Data_Preparation_and_RAG).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finetuning Transformer Models

##1 Finetuning BERT Model (Encoder-only).

Best for text classification and prediction

### 1.1 Data Preparation for BERT Finetuning

In [1]:
dataset = [("This movie is great", "Positive"),
           ("Not worth watching!", "Negative"),
           ("Amazing watch!", "Postive"),
           ("Boring watch with predicatable story line", "Negative")]

### 1.2 Load Bert Tokenizer and Tokenize Data

In [2]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

max_length = 128
formatted_data = [(f"[CLS] {text} [SEP]", label) for text, label in dataset]
tokenized_data = tokenizer(formatted_data, padding=True, truncation=True,
                  max_length=max_length,
                  return_tensors='pt')


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

### Convert tokenized, formatted text into numerical representations

In [3]:
import torch
from sklearn.preprocessing import LabelEncoder

input_ids = tokenized_data['input_ids']
attention_mask = tokenized_data['attention_mask']
labels = torch.tensor(LabelEncoder()
.fit_transform([label for _, label in dataset]))

### Split Initial dataset into training and validation sets

In [4]:
from sklearn.model_selection import train_test_split
train_inputs, val_inputs, train_labels, val_labels,train_mask, val_mask = train_test_split(
input_ids, labels, attention_mask,
random_state=42, test_size=0.1
)

Create custom dataset class

In [5]:
from torch.utils.data import Dataset
class CustomDataset(Dataset):
 def __init__(self, input_ids, attention_mask,labels):
  self.input_ids = input_ids
  self.attention_mask = attention_mask
  self.labels = labels
 def __len__(self):
  return len(self.input_ids)
 def __getitem__(self, idx):
  return {'input_ids': self.input_ids[idx],
          'attention_mask':self.attention_mask[idx],
          'labels': self.labels[idx]}

### Create data loaders

In [6]:
from torch.utils.data import DataLoader
batch_size = 4
train_dataset = CustomDataset(train_inputs, train_mask, train_labels)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True)
val_dataset = CustomDataset(val_inputs, val_mask, val_labels)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size,
                            shuffle=False)

### Load model from Hugging Face Hub

In [7]:
from transformers import BertForSequenceClassification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased',
 num_labels=len(set(labels)))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 2. Finetuning GPT model (decoder only)

Best for text generation

### 2.1 Data Preparation for BERT Finetuning

In [17]:
dataset = [("Once upon a time in a faraway swamp,",
 "there lived an ugly ork."),
           ("In the land of myth, in the time of magic, the destiny of a great kingdom rests on the shoulders of a young man", "his name: Merlin"),
           ("Beautiful People, Great Nation", "Nigeria my home!"),
]

### Load GPT tokenizer and tokenize data

In [18]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
formatted_data = [(f"[CLS] {context} [SEP] {target} [SEP]",)
for context, target in dataset]
numerical_data = [tokenizer.encode(example[0], add_special_tokens=True)
for example in formatted_data]

### Convert tokenized, formatted text into numerical representations

In [21]:
import torch

# Set pad_token_id for the tokenizer if it's not already set
# GPT2 tokenizer does not have a default pad_token_id
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

max_length = max(len(seq) for seq in numerical_data)
padded_data = [seq + [tokenizer.pad_token_id] * (max_length - len(seq))
for seq in numerical_data]

input_ids = torch.tensor(padded_data)

### Define custom dataset class

In [23]:
from torch.utils.data import Dataset
class CustomDataset(Dataset):
 def __init__(self, input_ids):
  self.input_ids = input_ids
 def __len__(self):
  return len(self.input_ids)
 def __getitem__(self, idx):
  return {'input_ids': self.input_ids[idx]}

### Create data loader

In [24]:
from torch.utils.data import DataLoader
batch_size = 4
custom_dataset = CustomDataset(input_ids)
dataloader = DataLoader(custom_dataset, batch_size=batch_size,
                        shuffle=True)

### Load model from Hugging Face Hub

In [25]:
from transformers import GPT2LMHeadModel
model = GPT2LMHeadModel.from_pretrained('gpt2')

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Data Preprocessing for RAG

Key importance for RAG: Knowledge base must be converted to embeddings and stored in a vector database

Library to be used for retrieving info from vector db is Facebook AI Similarity Search (FAISS, https://github.com/
facebookresearch/faiss)

Similarities between a vector and a query vector is gotten as the vector with the smallest Euclidean (L2) distance to query vector



In [26]:
data = [['His secret identity is Peter Parker', 'spiderman'],
 ['A businessman and engineer who ' +
 'runs the company Stark Industries',
 'ironman'],
 ['Superhuman spider-powers and abilities ' +
 'after being bitten by a radioactive spider',
 'spiderman'],
 ['A frail man enhanced to the peak of human ' +
 'physical perfection by an experimental super-soldier serum',
  'captainamerica']
 ]

In [27]:
import pandas as pd
df = pd.DataFrame(data, columns = ['text', 'context'])

Sentence transformers (https://**www**.sbert.net) is framework used to generate sentence, text and image embeddings

In [28]:
from sentence_transformers import SentenceTransformer
text = df['text']
encoder = SentenceTransformer("paraphrase-mpnet-base-v2")
vectors = encoder.encode(text.to_list())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.73k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Create FAISS index and add word embeddings to it

In [29]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 32.9 MB/s eta 0:00:00


In [30]:
import faiss
vector_dimension = vectors.shape[1]
l2_index = faiss.IndexFlatL2(vector_dimension)
faiss.normalize_L2(vectors)
l2_index.add(vectors)

Search Example

In [31]:
search_text = 'He throws webs'

import numpy as np
search_vector = encoder.encode(search_text)
search_vector_as_array = np.array([search_vector])
faiss.normalize_L2(search_vector_as_array)

# Perform search
k = l2_index.ntotal
distances, ann = l2_index.search(search_vector_as_array, k=k)

search_results = pd.DataFrame({'distances': distances[0], 'ann': ann[0]})
merged_df = pd.merge(search_results, df, left_on='ann', right_index=True)

In [32]:
merged_df

,distances,ann,text,context
0,1.501070,2,Superhuman spider-powers and abilities after b...,spiderman
1,1.552392,0,His secret identity is Peter Parker,spiderman
2,1.667212,1,A businessman and engineer who runs the compan...,ironman
3,1.731642,3,A frail man enhanced to the peak of human phys...,captainamerica
